In [1]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableSequence
import getpass
from collections.abc import Sequence, Mapping

class BookSummarizer:
    """Summarizes a book, providing both a narrative summary, and a bulleted list."""

    def __init__(
        self,
        llm: langchain_openai.chat_models.base.ChatOpenAI,
        chunk_size: int = 2500,
        chunk_overlap: int = 100,
    ) -> None:
        self._llm = llm
        self._text_splitter = TokenTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        self._cached_chain = self._get_chain()

    def _split_into_chunks(self, book_input: str) -> Sequence[Mapping[str, str]]:
        """Chunks text and returns a Sequence of Mappings."""
        return [
            {'chunk': chunk}
            for chunk in self._text_splitter.split_text(book_input)
        ]

    def _map_chain(self) -> RunnableSequence:
        """Summarize each chunk."""
        map_prompt_template = '''
        Write a concise summary of the following text, and include the main details.
        Text: {chunk}
        '''
        map_prompt = PromptTemplate.from_template(map_prompt_template)
        return map_prompt | self._llm | StrOutputParser()
        
    def _combine_summaries(self, summaries: Sequence[str]) -> Mapping[str, str]:
        """Combine summaries for each chunk."""
        return {'summaries': '\n'.join(summaries)}

    def _reduce_chain(self) -> RunnableSequence:
        """Provides a narrative summary of individual summaries."""
        reduce_prompt_template = '''
        Write a concise summary of the following text, which joins several summaries, and include the main details.
        Text: {summaries}
        '''
        reduce_prompt = PromptTemplate.from_template(reduce_prompt_template)
        return reduce_prompt | self._llm | StrOutputParser()

    def _bullet_chain(self) -> RunnableSequence:
        """Provides a bullet point summary of individual summaries."""
        bullet_prompt_template = '''
        List 5 key takeaways from these summaries.
        Text: {summaries}
        '''
        bullet_prompt = PromptTemplate.from_template(bullet_prompt_template)
        return bullet_prompt | self._llm | StrOutputParser()

    def _get_chain(self) -> RunnableSequence:
        """Assembles and returns the full LCEL chain."""
        return (
            RunnableLambda(self._split_into_chunks)
            | self._map_chain().map() 
            | RunnableLambda(self._combine_summaries)
            | RunnableParallel({
                'narrative_summary': self._reduce_chain(),
                'bullet_points': self._bullet_chain(),
            })
        )

    def summarize(self, book: str) -> Mapping[str, str]:
        """Summarize a book and return a narrative summary, as well as bullet points.

        Args:
            book: Input book to be summarized.
        Returns:
            The summary including:
                'narrative_summary': The narrative summary of the book.
                'bullet_points': Bullet point summary.
        """
        if not book:
            return {'narrative_summary': '', 'bullet_points': ''}
        return self._cached_chain.invoke(book)        


In [2]:
# Setup:
OPENAI_API_KEY = getpass.getpass('Enter your open AI API key:')
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name='gpt-5-nano',
)
bs = BookSummarizer(llm)

Enter your open AI API key: ········


In [3]:
# Read some input:
with open('docs/moby_dick_chapters_1-3.txt', 'r', encoding='utf-8') as fptr:
    book = fptr.read()

# Summarize.    
summary = bs.summarize(book)

In [4]:
print(summary['narrative_summary'])
print()
print('*-'*50)
print()
print(summary['bullet_points'])

- Ishmael, lonely and unsettled, seeks the sea to cure his gloom, viewing the voyage as fate-driven and drawn by the sea’s mystery, rather than a matter of free will; he plans a whaling voyage as part of a Providence-guided scheme. He leaves New York for Nantucket but ends up in New Bedford on a cold December night with little money.

- He wanders bleak streets, passing cheap lodgings (The Crossed Harpoons, The Sword-Fish Inn, The Trap) before finding shelter at The Spouter Inn in the care of Peter Coffin; the night is dark, windy, and pensive, with Ishmael contemplating the universe and life’s rough, unfinished state.

- The inn features a murky room and a large, unsettling oil painting of Cape Horn in a hurricane, with harpoons and weapons on the wall, and a dim bar where a whale’s jaw decorates the space; Ishmael notes the rough humor and the rough camaraderie he may share with sailors.

- A group of Grampus’s crew, including a lively landlord and the shipmate Bulkington, depart on 